# Класифікація супутникових знімків: Siamese Triplet Network
### Дипломна робота — порівняльний аналіз архітектур нейронних мереж
**Датасет:** NWPU-RESISC45 (Hugging Face `timm/resisc45`)  
**Класи:** 10 типів інфраструктурних об'єктів  
**Моделі:** Siamese Triplet (наша) | ResNet-50 Baseline | MobileNetV3 Baseline

---
> **Перед запуском:** увімкніть GPU — `Середовище виконання -> Змінити тип -> T4 GPU`  
> Запускайте клітинки суворо по порядку.

| Клітинка | Призначення |
|----------|-------------|
| 1 | Встановлення залежностей |
| 2 | Імпорти та налаштування |
| 3 | Конфігурація гіперпараметрів |
| 4 | Завантаження та розподіл датасету |
| 5 | Візуалізація прикладів датасету |
| 6 | Трансформації та DataLoader |
| 7 | Архітектури моделей |
| 8 | Навчання Siamese Triplet Network |
| 9 | Графіки навчання |
| 10 | Benchmark на тестовому наборі |
| 11 | Порівняльні графіки |
| 12 | Прототипи класів |
| 13 | Інтерактивний інтерфейс |
| 14 | Фінальний звіт |


In [ ]:
# Встановлення залежностей
%pip install -q torch torchvision datasets scikit-learn matplotlib Pillow ipywidgets scipy
from IPython.display import clear_output
clear_output()
print('Залежності встановлено.')


In [ ]:
# Імпорти та глобальні налаштування
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
matplotlib.rc('font', family='DejaVu Sans')

from PIL import Image, ImageDraw
from scipy.ndimage import label as scipy_label, binary_dilation
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files as colab_files
from datasets import load_dataset
from sklearn.metrics import average_precision_score

import io, os, random, time, warnings, datetime

warnings.filterwarnings('ignore')
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Пристрій: {DEVICE.upper()}')
if DEVICE == 'cpu':
    print('GPU не знайдено! Увімкніть: Середовище виконання -> Змінити тип -> T4 GPU')


In [ ]:
# Конфігурація
DATASET_PATH   = 'timm/resisc45'
BATCH_SIZE     = 32
LEARNING_RATE  = 1e-4
TRIPLET_MARGIN = 1.0
EMBED_DIM      = 128
NUM_EPOCHS     = 5  # для диплому рекомендується 10-15

TARGET_CLASS_NAMES = [
    'roundabout', 'intersection', 'industrial_area', 'medium_residential',
    'forest', 'lake', 'bridge', 'railway', 'stadium', 'runway'
]

CLASS_UKR = {
    'roundabout':         "Кільцева розв'язка",
    'intersection':       'Перехрестя доріг',
    'industrial_area':    'Промислова зона',
    'medium_residential': 'Житлова забудова',
    'forest':             'Ліс / Парк',
    'lake':               'Озеро / Водойма',
    'bridge':             'Міст',
    'railway':            'Залізниця',
    'stadium':            'Стадіон',
    'runway':             'Злітна смуга',
}

COLORS = {
    'siamese':   '#2ecc71',
    'resnet':    '#3498db',
    'mobilenet': '#e67e22',
}

MODEL_NAMES = {
    'siamese':   'Siamese Triplet (наша модель)',
    'resnet':    'ResNet-50 Baseline',
    'mobilenet': 'MobileNetV3 Baseline',
}


In [ ]:
# Завантаження та розподіл датасету
# Розподіл: 70% train / 10% val / 20% test
# Тестова вибірка ПОВНІСТЮ ІЗОЛЬОВАНА від навчання

raw_dataset    = load_dataset(DATASET_PATH, split='train')
all_classes    = raw_dataset.features['label'].names
target_indices = {all_classes.index(n): n for n in TARGET_CLASS_NAMES}
idx_to_name    = target_indices
name_to_idx    = {v: k for k, v in idx_to_name.items()}

filtered = raw_dataset.filter(lambda ex: ex['label'] in target_indices)

split1     = filtered.train_test_split(test_size=0.2, seed=42)
split2     = split1['train'].train_test_split(test_size=0.125, seed=42)
train_data = split2['train']
val_data   = split2['test']
test_data  = split1['test']  # ІЗОЛЬОВАНА тестова вибірка

train_labels = train_data['label']
val_labels   = val_data['label']
test_labels  = test_data['label']

print(f'Train:      {len(train_data):>6,} зображень')
print(f'Validation: {len(val_data):>6,} зображень')
print(f'Test:       {len(test_data):>6,} зображень  <- ізольована')
print(f'Всього:     {len(filtered):>6,} зображень')


In [ ]:
# Візуалізація прикладів датасету
fig, axs = plt.subplots(2, 5, figsize=(18, 9))
axs = axs.ravel()
fig.suptitle('RESISC45 - приклади зображень (навчальна вибірка)',
             fontsize=14, fontweight='bold', y=1.01)

for pos, cls_idx in enumerate(sorted(idx_to_name.keys())):
    sample = next(it for it in train_data if it['label'] == cls_idx)
    axs[pos].imshow(sample['image'])
    eng = idx_to_name[cls_idx]
    axs[pos].set_title(f"{CLASS_UKR[eng]}\n({eng})",
                       fontsize=9, fontweight='bold', pad=8)
    axs[pos].axis('off')

plt.tight_layout(pad=2.0, h_pad=3.0, w_pad=1.5)
plt.savefig('01_dataset_preview.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Трансформації та DataLoader

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class TripletDataset(Dataset):
    """Повертає трійки (anchor, positive, negative) для Triplet Loss."""

    def __init__(self, hf_dataset, labels_list, transform=None):
        self.ds = hf_dataset
        self.labels = labels_list
        self.transform = transform
        self.class_indices = {}
        for i, lbl in enumerate(self.labels):
            self.class_indices.setdefault(lbl, []).append(i)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        lbl  = self.labels[idx]
        anc  = self.ds[idx]['image'].convert('RGB')
        pool = [i for i in self.class_indices[lbl] if i != idx]
        pos  = self.ds[random.choice(pool) if pool else idx]['image'].convert('RGB')
        nc   = random.choice([c for c in self.class_indices if c != lbl])
        neg  = self.ds[random.choice(self.class_indices[nc])]['image'].convert('RGB')
        if self.transform:
            anc, pos, neg = self.transform(anc), self.transform(pos), self.transform(neg)
        return anc, pos, neg, lbl


class SingleDataset(Dataset):
    """Звичайний датасет (зображення + мітка) для оцінювання."""

    def __init__(self, hf_dataset, labels_list, transform=None):
        self.ds = hf_dataset
        self.labels = labels_list
        self.transform = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        img = self.ds[idx]['image'].convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


train_loader = DataLoader(
    TripletDataset(train_data, train_labels, transform_train),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    SingleDataset(val_data, val_labels, transform_test),
    batch_size=64, shuffle=False, num_workers=2,
)
test_loader = DataLoader(
    SingleDataset(test_data, test_labels, transform_test),
    batch_size=64, shuffle=False, num_workers=2,
)

print(f'Train: {len(train_loader)} батчів | Val: {len(val_loader)} | Test: {len(test_loader)}')


In [ ]:
# Визначення архітектур моделей


class SiameseTripletNetwork(nn.Module):
    """
    Наша модель: ResNet-50 backbone + Embedding Head навчений з Triplet Loss.

    Backbone : ResNet-50 (ImageNet pretrained) без FC-шару -> вектор 2048
    Head     : Linear(2048->512) -> BN -> ReLU -> Dropout(0.3) -> Linear(512->128)
    Вихід    : L2-нормалізований ембеддинг розміром EMBED_DIM
    """

    def __init__(self, embed_dim: int = EMBED_DIM):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.head = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, embed_dim),
        )

    def forward_once(self, x):
        return nn.functional.normalize(
            self.head(self.backbone(x).flatten(1)), p=2, dim=1
        )

    def forward(self, a, p, n):
        return self.forward_once(a), self.forward_once(p), self.forward_once(n)


class BaselineResNet50(nn.Module):
    """Baseline: ResNet-50 без triplet fine-tuning (заморожені ваги ImageNet)."""

    def __init__(self):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.extractor = nn.Sequential(*list(base.children())[:-1])

    def forward(self, x):
        return nn.functional.normalize(self.extractor(x).flatten(1), p=2, dim=1)


class BaselineMobileNet(nn.Module):
    """Baseline: MobileNetV3-Small (заморожені ваги ImageNet)."""

    def __init__(self):
        super().__init__()
        base = models.mobilenet_v3_small(
            weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
        )
        self.extractor = nn.Sequential(*list(base.children())[:-1])

    def forward(self, x):
        return nn.functional.normalize(self.extractor(x).flatten(1), p=2, dim=1)


siamese_model   = SiameseTripletNetwork(embed_dim=EMBED_DIM).to(DEVICE)
baseline_resnet = BaselineResNet50().to(DEVICE).eval()
baseline_mobile = BaselineMobileNet().to(DEVICE).eval()

n_params = sum(p.numel() for p in siamese_model.parameters() if p.requires_grad)
print(f'Siamese Triplet: {n_params / 1e6:.1f}M параметрів на [{DEVICE.upper()}]')
print('ResNet-50 Baseline  : заморожений')
print('MobileNetV3 Baseline: заморожений')


In [ ]:
# Навчання Siamese Triplet Network

criterion = nn.TripletMarginLoss(margin=TRIPLET_MARGIN, p=2)
optimizer = torch.optim.AdamW(
    siamese_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS
)
history = {'loss': [], 'val_map': []}


def compute_map(model, loader, n=300):
    """Обчислює mAP на підмножині вибірки."""
    model.eval()
    embs, lbls = [], []
    with torch.no_grad():
        for imgs, ls in loader:
            e = (
                model.forward_once(imgs.to(DEVICE))
                if hasattr(model, 'forward_once')
                else model(imgs.to(DEVICE))
            )
            embs.append(e.cpu().numpy())
            lbls.extend(ls.tolist())
            if len(lbls) >= n:
                break
    embs = np.vstack(embs)[:n]
    lbls = np.array(lbls[:n])
    sim  = embs @ embs.T
    aps  = []
    for i in range(len(lbls)):
        gt = (lbls == lbls[i]).astype(int)
        gt[i] = 0
        sc = sim[i].copy()
        sc[i] = -1
        if gt.sum() > 0:
            aps.append(average_precision_score(gt, sc))
    return float(np.mean(aps)) * 100 if aps else 0.0


best_val, best_epoch = 0.0, 0

for epoch in range(1, NUM_EPOCHS + 1):
    siamese_model.train()
    run_loss = 0.0
    t0 = time.time()

    for bi, (anc, pos, neg, _) in enumerate(train_loader):
        anc, pos, neg = anc.to(DEVICE), pos.to(DEVICE), neg.to(DEVICE)
        optimizer.zero_grad()
        ea, ep, en = siamese_model(anc, pos, neg)
        loss = criterion(ea, ep, en)
        loss.backward()
        nn.utils.clip_grad_norm_(siamese_model.parameters(), 1.0)
        optimizer.step()
        run_loss += loss.item()
        if bi % 25 == 0:
            print(
                f'  Epoch {epoch}/{NUM_EPOCHS} | '
                f'Batch {bi:3d}/{len(train_loader)} | '
                f'Loss: {loss.item():.4f}',
                end='\r'
            )

    avg_loss = run_loss / len(train_loader)
    val_map  = compute_map(siamese_model, val_loader)
    history['loss'].append(avg_loss)
    history['val_map'].append(val_map)
    scheduler.step()

    if val_map > best_val:
        best_val, best_epoch = val_map, epoch
        torch.save(siamese_model.state_dict(), 'best_siamese_model.pth')

    elapsed = time.time() - t0
    print(
        f'  Epoch {epoch}/{NUM_EPOCHS} | '
        f'Loss: {avg_loss:.4f} | '
        f'Val mAP: {val_map:.1f}% | '
        f'{elapsed:.0f}s   '
    )

siamese_model.load_state_dict(torch.load('best_siamese_model.pth', map_location=DEVICE))
siamese_model.eval()
print(f'Навчання завершено. Найкращий Val mAP: {best_val:.1f}% (epoch {best_epoch})')
print('Ваги збережено: best_siamese_model.pth')


In [ ]:
# Графіки динаміки навчання
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Динаміка навчання - Siamese Triplet Network',
             fontsize=13, fontweight='bold')

ep = range(1, len(history['loss']) + 1)

ax1.plot(ep, history['loss'], 'o-', color='#e74c3c', lw=2, ms=5)
ax1.fill_between(ep, history['loss'], alpha=0.1, color='#e74c3c')
ax1.set(title='Triplet Loss (навчання)', xlabel='Епоха', ylabel='Loss')
ax1.grid(True, alpha=0.4)

ax2.plot(ep, history['val_map'], 's-', color='#2ecc71', lw=2, ms=5)
ax2.fill_between(ep, history['val_map'], alpha=0.1, color='#2ecc71')
ax2.set(title='mAP (валідація)', xlabel='Епоха', ylabel='mAP (%)')
ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('02_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Benchmark на ізольованому тестовому наборі


def full_eval(model, loader, key, n=500):
    """Повна оцінка: mAP, Top-1 accuracy, час інференсу, AP по класах."""
    model.eval()
    embs, lbls = [], []
    t0 = time.time()

    with torch.no_grad():
        for imgs, ls in loader:
            e = (
                model.forward_once(imgs.to(DEVICE))
                if hasattr(model, 'forward_once')
                else model(imgs.to(DEVICE))
            )
            embs.append(e.cpu().numpy())
            lbls.extend(ls.tolist())
            if len(lbls) >= n:
                break

    t_ms = (time.time() - t0) / len(lbls) * 1000
    embs = np.vstack(embs)[:n]
    lbls = np.array(lbls[:n])
    sim  = embs @ embs.T

    aps = []
    for i in range(len(lbls)):
        gt = (lbls == lbls[i]).astype(int)
        gt[i] = 0
        sc = sim[i].copy()
        sc[i] = -1
        if gt.sum() > 0:
            aps.append(average_precision_score(gt, sc))
    mAP = float(np.mean(aps)) * 100 if aps else 0.0

    top1 = 0
    for i in range(len(lbls)):
        sc = sim[i].copy()
        sc[i] = -999
        if lbls[np.argmax(sc)] == lbls[i]:
            top1 += 1
    top1 = top1 / len(lbls) * 100

    pc = {}
    for ci in np.unique(lbls):
        gt = (lbls == ci).astype(int)
        ms = sim[:, lbls == ci].mean(axis=1)
        if gt.sum() > 0:
            pc[ci] = average_precision_score(gt, ms) * 100

    print(f'  {MODEL_NAMES[key]:32s}  mAP:{mAP:5.1f}%  Top-1:{top1:5.1f}%  {t_ms:.1f} ms/img')
    return {'mAP': mAP, 'top1': top1, 'time': t_ms, 'per_class': pc}


bench = {}
print(f"  {'Модель':32s}  {'mAP':>7}  {'Top-1':>7}  {'Час':>10}")
print('  ' + '-' * 58)
bench['siamese']   = full_eval(siamese_model,   test_loader, 'siamese')
bench['resnet']    = full_eval(baseline_resnet, test_loader, 'resnet')
bench['mobilenet'] = full_eval(baseline_mobile, test_loader, 'mobilenet')


In [ ]:
# Порівняльні графіки трьох моделей
keys  = ['siamese', 'resnet', 'mobilenet']
clrs  = [COLORS[k] for k in keys]
maps  = [bench[k]['mAP']  for k in keys]
top1s = [bench[k]['top1'] for k in keys]
times = [bench[k]['time'] for k in keys]
short = ['Siamese\n(наша)', 'ResNet-50', 'MobileNetV3']

fig = plt.figure(figsize=(16, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('Порівняння моделей - ізольований тестовий набір',
             fontsize=14, fontweight='bold')

for col, (vals, title, ylabel, annotate) in enumerate([
    (maps,  'Середня точність (mAP)',        'mAP (%)',          True),
    (top1s, 'Точність Top-1 (1-NN)',         'Точність (%)',     False),
    (times, 'Час інференсу (менше = краще)', 'мс / зображення', False),
]):
    ax = fig.add_subplot(gs[0, col])
    bars = ax.bar(range(3), vals, color=clrs, edgecolor='white', width=0.55)
    ax.set_xticks(range(3))
    ax.set_xticklabels(short, fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ylim = 105 if col < 2 else max(times) * 1.3
    ax.set_ylim(0, ylim)
    for bar, v in zip(bars, vals):
        suf = '%' if col < 2 else ' мс'
        ax.text(bar.get_x() + bar.get_width() / 2, v + ylim * 0.015,
                f'{v:.1f}{suf}', ha='center', fontsize=10, fontweight='bold')
    if annotate:
        bi = int(np.argmax(vals))
        ax.annotate('Найкраща', xy=(bi, vals[bi]),
                    xytext=(bi, vals[bi] + ylim * 0.1),
                    ha='center', fontsize=9, color='#27ae60',
                    arrowprops=dict(arrowstyle='->', color='#27ae60', lw=1.5))

ax4 = fig.add_subplot(gs[1, :2])
pc       = bench['siamese']['per_class']
cls_lbls = [CLASS_UKR[idx_to_name[i]] for i in sorted(pc)]
cls_vals = [pc[i] for i in sorted(pc)]
clrs4    = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(cls_lbls)))
bars4    = ax4.barh(cls_lbls, cls_vals, color=clrs4, edgecolor='white')
ax4.set_xlabel('Середня точність (%)')
ax4.set_title('Точність по класах - Siamese Triplet (наша модель)', fontweight='bold')
ax4.set_xlim(0, 108)
ax4.grid(axis='x', alpha=0.3)
for bar, v in zip(bars4, cls_vals):
    ax4.text(v + 1, bar.get_y() + bar.get_height() / 2,
             f'{v:.1f}%', va='center', fontsize=9)

ax5 = fig.add_subplot(gs[1, 2], polar=True)
N      = 3
angles = [n / N * 2 * 3.14159 for n in range(N)] + [0]
mt     = max(times)
radar  = {k: [bench[k]['mAP'], bench[k]['top1'],
              (1 - bench[k]['time'] / mt) * 100] for k in keys}
for k, nm, c in zip(keys, short, clrs):
    v = radar[k] + [radar[k][0]]
    ax5.plot(angles, v, 'o-', lw=2, color=c, label=nm.replace('\n', ' '))
    ax5.fill(angles, v, alpha=0.08, color=c)
ax5.set_xticks(angles[:-1])
ax5.set_xticklabels(['mAP', 'Top-1', 'Швидкість\n(інверс.)'], fontsize=8)
ax5.set_ylim(0, 110)
ax5.set_title('Профіль моделей', fontweight='bold', fontsize=9, pad=15)
ax5.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=7)

plt.savefig('03_comparison_charts.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Побудова прототипів класів


def get_emb(model, img_pil):
    """Повертає L2-нормалізований ембеддинг зображення."""
    t = transform_test(img_pil.convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        e = model.forward_once(t) if hasattr(model, 'forward_once') else model(t)
    return e.cpu().numpy()[0]


def build_prototypes(model, n=15):
    """Будує усереднені прототипи класів з навчальної вибірки."""
    prototypes = {}
    for ci in idx_to_name:
        imgs  = [it['image'] for it in train_data if it['label'] == ci][:n]
        embs  = [get_emb(model, img) for img in imgs]
        proto = np.mean(embs, axis=0)
        prototypes[ci] = proto / (np.linalg.norm(proto) + 1e-8)
    return prototypes


def classify(img_pil):
    """Класифікує зображення трьома моделями."""
    out = {}
    for tag, model, protos in [
        ('siamese',   siamese_model,   proto_siamese),
        ('resnet',    baseline_resnet, proto_resnet),
        ('mobilenet', baseline_mobile, proto_mobilenet),
    ]:
        emb  = get_emb(model, img_pil)
        sims = {ci: float(np.dot(emb / (np.linalg.norm(emb) + 1e-8), proto))
                for ci, proto in protos.items()}
        out[tag] = sorted(sims.items(), key=lambda x: x[1], reverse=True)
    return out


print('Будуємо прототипи класів...')
proto_siamese   = build_prototypes(siamese_model)
proto_resnet    = build_prototypes(baseline_resnet)
proto_mobilenet = build_prototypes(baseline_mobile)
print('Прототипи готові.')


In [ ]:
# Інтерактивний інтерфейс детекції

_uploaded_img  = None
_generated_map = None

CLASS_COLORS_RGB = {
    'roundabout':         (255,  80,  80),
    'intersection':       (255, 165,   0),
    'industrial_area':    (180, 180, 180),
    'medium_residential': (255, 220,  50),
    'forest':             ( 50, 200,  80),
    'lake':               ( 50, 150, 255),
    'bridge':             (200, 100,  50),
    'railway':            (180,  50, 255),
    'stadium':            (  0, 220, 220),
    'runway':             (230, 230, 230),
}

CLASS_UKR_FULL = {
    'roundabout':         "Кільцева розв'язка",
    'intersection':       'Перехрестя доріг',
    'industrial_area':    'Промислова зона',
    'medium_residential': 'Житлова забудова',
    'forest':             'Ліс / Парк',
    'lake':               'Озеро / Водойма',
    'bridge':             'Міст',
    'railway':            'Залізниця',
    'stadium':            'Стадіон',
    'runway':             'Злітна смуга',
}

CLASS_THRESHOLDS = {
    'roundabout':         0.30,
    'intersection':       0.30,
    'industrial_area':    0.48,
    'medium_residential': 0.48,
    'forest':             0.48,
    'lake':               0.48,
    'bridge':             0.36,
    'railway':            0.36,
    'stadium':            0.48,
    'runway':             0.42,
}

MODEL_THRESHOLD_SCALE = {'siamese': 1.0, 'resnet': 1.4, 'mobilenet': 1.5}


def generate_mosaic_map(grid=4):
    """Генерує мозаїчну карту з test set."""
    size   = 224
    canvas = Image.new('RGB', (size * grid, size * grid))
    items  = list(range(len(test_data)))
    random.shuffle(items)
    labels = []
    for i, idx in enumerate(items[:grid * grid]):
        row, col = divmod(i, grid)
        img = test_data[idx]['image'].convert('RGB').resize((size, size))
        canvas.paste(img, (col * size, row * size))
        labels.append(test_data[idx]['label'])
    return canvas, labels


def compute_iou(a, b):
    """Intersection over Union між двома bounding boxes."""
    ix1 = max(a[1], b[1]); iy1 = max(a[2], b[2])
    ix2 = min(a[3], b[3]); iy2 = min(a[4], b[4])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area  = lambda box: max(0, box[3] - box[1]) * max(0, box[4] - box[2])
    union = area(a) + area(b) - inter
    return inter / union if union > 0 else 0.0


def nms(boxes, iou_thr=0.35):
    """Non-Maximum Suppression."""
    if not boxes:
        return []
    boxes = sorted(boxes, key=lambda x: x[0], reverse=True)
    kept  = []
    while boxes:
        best = boxes.pop(0)
        kept.append(best)
        boxes = [b for b in boxes if compute_iou(best, b) < iou_thr]
    return kept


def build_heatmap(img_pil, model_obj, proto_dict, target_class_indices, grid=None):
    """
    Multi-scale sliding window детекція.

    Вікно 224x224 (розмір навчальних зображень) при 5 масштабах:
    0.3x, 0.5x, 0.75x, 1.0x, 1.5x. Крок 50%. NMS threshold=0.35.
    """
    W, H   = img_pil.size
    WIN    = 224
    STEP   = WIN // 2
    SCALES = [0.3, 0.5, 0.75, 1.0, 1.5]

    all_detections = {ci: [] for ci in target_class_indices}

    for scale in SCALES:
        new_W = max(WIN, int(W * scale))
        new_H = max(WIN, int(H * scale))
        img_s = img_pil.resize((new_W, new_H), Image.BILINEAR)

        xs = list(range(0, new_W - WIN + 1, STEP)) or [0]
        ys = list(range(0, new_H - WIN + 1, STEP)) or [0]
        if xs[-1] + WIN < new_W: xs.append(new_W - WIN)
        if ys[-1] + WIN < new_H: ys.append(new_H - WIN)

        for y in ys:
            for x in xs:
                patch    = img_s.crop((x, y, x + WIN, y + WIN))
                emb      = get_emb(model_obj, patch)
                emb_norm = emb / (np.linalg.norm(emb) + 1e-8)
                for ci in target_class_indices:
                    if ci not in proto_dict:
                        continue
                    score = float(np.dot(emb_norm, proto_dict[ci]))
                    ox1 = max(0, min(int(x / scale), W))
                    oy1 = max(0, min(int(y / scale), H))
                    ox2 = max(0, min(int((x + WIN) / scale), W))
                    oy2 = max(0, min(int((y + WIN) / scale), H))
                    all_detections[ci].append((score, ox1, oy1, ox2, oy2, scale))

    result = img_pil.copy().convert('RGB')
    draw   = ImageDraw.Draw(result)
    found_per_class = {}
    patch_results   = []
    heatmaps        = {}

    for ci in target_class_indices:
        eng     = idx_to_name.get(ci, '')
        r, g, b = CLASS_COLORS_RGB.get(eng, (0, 220, 80))
        dets    = all_detections[ci]
        if not dets:
            continue

        max_score = max(d[0] for d in dets)
        threshold = max(CLASS_THRESHOLDS.get(eng, 0.30), max_score * 0.88)
        candidates = [d for d in dets if d[0] >= threshold]
        if not candidates:
            if max_score >= 0.25:
                candidates = [max(dets, key=lambda x: x[0])]
            else:
                continue

        kept = nms(candidates, iou_thr=0.35)[:3]

        for rank, det in enumerate(kept):
            score, x1, y1, x2, y2, _ = det
            result_np = np.array(result, dtype=np.float32)
            alpha     = 0.15 if rank == 0 else 0.05
            result_np[y1:y2, x1:x2] = (
                result_np[y1:y2, x1:x2] * (1 - alpha)
                + np.array([r, g, b], dtype=np.float32) * alpha
            )
            result = Image.fromarray(result_np.clip(0, 255).astype(np.uint8))
            draw   = ImageDraw.Draw(result)
            box_size = max(x2 - x1, y2 - y1)
            thick    = max(3, box_size // 80) if rank == 0 else 2
            for t in range(thick):
                draw.rectangle([x1 + t, y1 + t, x2 - t, y2 - t], outline=(r, g, b))
            if rank == 0:
                label_txt = f"{eng}  {score * 100:.0f}%"
                lw = len(label_txt) * 8 + 10
                ly = max(0, y1 - 24)
                draw.rectangle([x1, ly, x1 + lw, ly + 22], fill=(r, g, b))
                draw.text((x1 + 5, ly + 4), label_txt, fill=(0, 0, 0))

        best = kept[0]
        found_per_class[ci] = len(kept)
        patch_results.append({
            'best_class': ci, 'confidence': best[0], 'detected': True,
            'x1': best[1], 'y1': best[2], 'x2': best[3], 'y2': best[4],
        })
        hm_size = 16
        hm = np.zeros((hm_size, hm_size))
        for sd, x1d, y1d, *_ in dets:
            mx = min(int(x1d / W * hm_size), hm_size - 1)
            my = min(int(y1d / H * hm_size), hm_size - 1)
            hm[my, mx] = max(hm[my, mx], sd)
        heatmaps[ci] = hm

    return result, patch_results, found_per_class, heatmaps


def summarize_patches(patch_results):
    found, conf_per_class = {}, {}
    for p in patch_results:
        if p['detected']:
            ci = p['best_class']
            found[ci] = found.get(ci, 0) + 1
            conf_per_class.setdefault(ci, []).append(p['confidence'])
    return found, {ci: float(np.mean(v)) for ci, v in conf_per_class.items()}




# ── Widgets ──────────────────────────────────────────────────
header_w = widgets.HTML(
    '<div style="background:#0f2027;padding:12px 20px;border-radius:10px;'
    'border-left:5px solid #2ecc71;margin-bottom:6px;">'
    '<h2 style="color:#ecf0f1;margin:0 0 3px;font-size:17px;font-family:sans-serif;">'
    'Аналізатор супутникових знімків</h2>'
    '<span style="color:#7f8c8d;font-size:11px;font-family:sans-serif;">'
    'Siamese Triplet | ResNet-50 | MobileNetV3'
    '</span></div>'
)

def sep_line():
    return widgets.HTML("<hr style='border:0;border-top:1px solid #2c3e50;margin:8px 0'>")

step1_lbl = widgets.HTML("<b style='font-size:13px;'>Крок 1 &nbsp; Оберіть режим:</b>")
mode_btn  = widgets.ToggleButtons(
    options=['Завантажити власне фото', 'Згенерувати тестову карту'],
    style={'button_width': '230px', 'font_size': '13px'},
    layout=widgets.Layout(margin='4px 0 6px'),
)

upload_widget  = widgets.FileUpload(accept='image/*', multiple=False,
                                    description='Вибрати фото',
                                    layout=widgets.Layout(width='180px', height='36px'))
upload_status  = widgets.HTML("<span style='color:#888;font-size:12px;'>Файл не вибрано</span>")
upload_preview = widgets.Output(layout=widgets.Layout(width='320px', height='250px'))
upload_block   = widgets.VBox([
    widgets.HTML("<b style='font-size:12px;'>Завантажте зображення (JPG/PNG/WEBP):</b>"),
    upload_widget, upload_status, upload_preview,
], layout=widgets.Layout(margin='4px 0'))

gen_btn     = widgets.Button(description='Згенерувати карту',
                              layout=widgets.Layout(width='200px', height='36px'),
                              style={'button_color': '#8e44ad', 'font_size': '13px'})
gen_status  = widgets.HTML("<span style='color:#888;font-size:12px;'>Карту не згенеровано</span>")
gen_preview = widgets.Output(layout=widgets.Layout(width='320px', height='250px'))
gen_block   = widgets.VBox([
    widgets.HTML("<b style='font-size:12px;'>Мозаїчна карта з test set:</b>"),
    gen_btn, gen_status, gen_preview,
], layout=widgets.Layout(display='none', margin='4px 0'))

step2_lbl = widgets.HTML(
    "<b style='font-size:13px;'>Крок 2 &nbsp; Які об'єкти шукати?</b>"
    "<span style='color:#888;font-size:11px;'> (кожен клас - свій колір рамки)</span>"
)

checkboxes, checkbox_rows = {}, []
for eng, ukr in CLASS_UKR_FULL.items():
    rc, gc, bc = CLASS_COLORS_RGB.get(eng, (200, 200, 200))
    color_hex = f'#{rc:02x}{gc:02x}{bc:02x}'
    cb  = widgets.Checkbox(value=False, description=ukr, indent=False,
                           layout=widgets.Layout(width='210px'),
                           style={'description_width': '0px'})
    dot = widgets.HTML(
        f"<div style='width:15px;height:15px;border-radius:3px;"
        f"background:{color_hex};margin:3px 6px 0 2px;"
        f"border:1px solid #555;flex-shrink:0;'></div>"
    )
    checkboxes[eng] = cb
    checkbox_rows.append(widgets.HBox([dot, cb],
                         layout=widgets.Layout(align_items='center')))

cb_row = widgets.HBox([widgets.VBox(checkbox_rows[:5]),
                       widgets.VBox(checkbox_rows[5:])])
sel_all_btn = widgets.Button(description='Вибрати всі',
                              layout=widgets.Layout(width='115px', height='28px'),
                              style={'font_size': '11px'})
desel_btn   = widgets.Button(description='Зняти всі',
                              layout=widgets.Layout(width='115px', height='28px'),
                              style={'font_size': '11px'})
sel_all_btn.on_click(lambda b: [setattr(cb, 'value', True)  for cb in checkboxes.values()])
desel_btn.on_click(  lambda b: [setattr(cb, 'value', False) for cb in checkboxes.values()])
cb_btns = widgets.HBox([sel_all_btn, desel_btn],
                        layout=widgets.Layout(margin='4px 0 0'))

step3_lbl  = widgets.HTML("<b style='font-size:13px;'>Крок 3 &nbsp;</b>")
run_btn    = widgets.Button(description='Запустити аналіз',
                             layout=widgets.Layout(width='200px', height='42px'),
                             style={'button_color': '#27ae60',
                                    'font_weight': 'bold', 'font_size': '14px'})
run_status = widgets.HTML('')


def model_label(text, color):
    return widgets.HTML(
        f"<div style='font-size:13px;font-weight:bold;color:{color};"
        f"padding:4px 0 2px;border-bottom:2px solid {color};margin-bottom:4px;'>"
        f"{text}</div>"
    )


lbl_s = model_label('Siamese Triplet (наша модель)', '#2ecc71')
lbl_r = model_label('ResNet-50 Baseline',            '#3498db')
lbl_m = model_label('MobileNetV3 Baseline',          '#e67e22')
img_out_s = widgets.Output()
img_out_r = widgets.Output()
img_out_m = widgets.Output()

images_row = widgets.VBox([
    widgets.VBox([lbl_s, img_out_s],
                 layout=widgets.Layout(width='100%', margin='0 0 20px 0')),
    widgets.VBox([lbl_r, img_out_r],
                 layout=widgets.Layout(width='100%', margin='0 0 20px 0')),
    widgets.VBox([lbl_m, img_out_m],
                 layout=widgets.Layout(width='100%', margin='0 0 20px 0')),
])

stats_out  = widgets.Output()
report_out = widgets.Output()
tab = widgets.Tab(children=[
    images_row,
    widgets.VBox([stats_out]),
    widgets.VBox([report_out]),
])
tab.set_title(0, 'Візуалізація')
tab.set_title(1, 'Статистика')
tab.set_title(2, 'Текстовий звіт')


def make_dl(label, fname):
    btn = widgets.Button(description=label,
                         layout=widgets.Layout(width='240px', height='32px'),
                         style={'font_size': '11px'})
    def click(b):
        if os.path.exists(fname):
            colab_files.download(fname)
        else:
            with stats_out:
                print(f'Файл {fname} ще не існує.')
    btn.on_click(click)
    return btn


dl_box = widgets.VBox([
    widgets.HTML("<b style='font-size:12px;'>Скачати на комп'ютер:</b>"),
    widgets.HBox([make_dl('Огляд датасету',      '01_dataset_preview.png'),
                  make_dl('Криві навчання',       '02_training_curves.png')]),
    widgets.HBox([make_dl('Порівняння моделей',  '03_comparison_charts.png'),
                  make_dl('Результат Siamese',   'preview_siamese.png')]),
    widgets.HBox([make_dl('Результат ResNet-50', 'preview_resnet.png'),
                  make_dl('Результат MobileNet', 'preview_mobilenet.png')]),
    widgets.HBox([make_dl('Текстовий звіт',      'diploma_report.txt'),
                  make_dl('Ваги моделі (.pth)',  'best_siamese_model.pth')]),
])


# ── Events ───────────────────────────────────────────────────
def on_mode_change(change):
    if mode_btn.index == 0:
        upload_block.layout.display = ''
        gen_block.layout.display    = 'none'
    else:
        upload_block.layout.display = 'none'
        gen_block.layout.display    = ''


mode_btn.observe(on_mode_change, names='value')


def on_file_upload(change):
    global _uploaded_img
    try:
        val = upload_widget.value
        if not val:
            return
        if isinstance(val, dict):
            content = list(val.values())[0]['content']
            fname   = list(val.keys())[0]
        else:
            content = val[0]['content']
            fname   = val[0]['name']
        _uploaded_img = Image.open(io.BytesIO(bytes(content))).convert('RGB')
        w, h = _uploaded_img.size
        upload_status.value = (
            f"<span style='color:#2ecc71;font-size:12px;'>"
            f"Завантажено: <b>{fname}</b> ({w}x{h}px)</span>"
        )
        with upload_preview:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow(_uploaded_img); ax.axis('off')
            ax.set_title('Попередній перегляд', fontsize=9)
            plt.tight_layout(pad=0.2); plt.show()
    except Exception as ex:
        upload_status.value = f"<span style='color:#e74c3c;font-size:12px;'>Помилка: {ex}</span>"


upload_widget.observe(on_file_upload, names='value')


def on_generate(b):
    global _generated_map
    gen_btn.disabled = True
    gen_status.value = "<span style='color:#f39c12;font-size:12px;'>Генерую карту...</span>"
    try:
        _generated_map, used_labels = generate_mosaic_map(grid=4)
        unique_ukr = list({CLASS_UKR_FULL.get(idx_to_name.get(l, ''), '?') for l in used_labels})
        gen_status.value = (
            f"<span style='color:#2ecc71;font-size:12px;'>"
            f"Карту згенеровано. Класи: {', '.join(unique_ukr[:5])}</span>"
        )
        with gen_preview:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow(_generated_map); ax.axis('off')
            ax.set_title('Мозаїчна карта (test set)', fontsize=9)
            plt.tight_layout(pad=0.2); plt.show()
    except Exception as ex:
        gen_status.value = f"<span style='color:#e74c3c;font-size:12px;'>Помилка: {ex}</span>"
    finally:
        gen_btn.disabled = False


gen_btn.on_click(on_generate)


def on_run(b):
    img_pil = _uploaded_img if mode_btn.index == 0 else _generated_map
    if img_pil is None:
        run_status.value = (
            "<span style='color:#e74c3c;font-size:12px;'>"
            "Спочатку завантажте фото або згенеруйте карту.</span>"
        )
        return

    target_names   = [eng for eng, cb in checkboxes.items() if cb.value]
    target_indices = {name_to_idx[n] for n in target_names if n in name_to_idx}
    if not target_indices:
        run_status.value = (
            "<span style='color:#e74c3c;font-size:12px;'>"
            "Оберіть хоча б один клас для пошуку.</span>"
        )
        return

    run_btn.disabled    = True
    run_btn.description = 'Аналізую...'
    run_status.value    = "<span style='color:#f39c12;font-size:12px;'>Обробка...</span>"

    try:
        model_objects = [siamese_model,   baseline_resnet, baseline_mobile]
        proto_dicts   = [proto_siamese,   proto_resnet,    proto_mobilenet]
        tags          = ['siamese',        'resnet',        'mobilenet']
        tag_names     = ['Siamese Triplet (наша)', 'ResNet-50 Baseline', 'MobileNetV3']
        tag_colors    = ['#2ecc71',                '#3498db',             '#e67e22']
        outs          = [img_out_s, img_out_r, img_out_m]
        all_found, all_avg_conf = {}, {}

        for tag, model_obj, proto_dict, out_w, tag_col in zip(
                tags, model_objects, proto_dicts, outs, tag_colors):
            scale  = MODEL_THRESHOLD_SCALE[tag]
            scaled = {eng: min(0.95, val * scale) for eng, val in CLASS_THRESHOLDS.items()}
            orig   = CLASS_THRESHOLDS.copy()
            CLASS_THRESHOLDS.update(scaled)

            annotated, patch_results, found, _ = build_heatmap(
                img_pil, model_obj, proto_dict, target_indices
            )
            CLASS_THRESHOLDS.update(orig)

            _, avg_conf       = summarize_patches(patch_results)
            all_found[tag]    = found
            all_avg_conf[tag] = avg_conf

            with out_w:
                clear_output(wait=True)
                W2, H2 = annotated.size
                fig, ax = plt.subplots(figsize=(11, 11 * H2 / W2))
                ax.imshow(annotated); ax.axis('off')
                if found:
                    lines = [
                        f"{CLASS_UKR_FULL.get(idx_to_name.get(ci,''),'?')}: "
                        f"{cnt} зон | {avg_conf.get(ci,0)*100:.0f}%"
                        for ci, cnt in found.items()
                    ]
                    ax.set_title('\n'.join(lines), fontsize=10, pad=6, color=tag_col)
                else:
                    ax.set_title('Нічого не знайдено', fontsize=10,
                                 pad=6, color='#e74c3c')
                plt.tight_layout(pad=0.1)
                plt.savefig(f'preview_{tag}.png', dpi=130, bbox_inches='tight')
                plt.show()

        with stats_out:
            clear_output(wait=True)
            tags_nm = list(zip(tags, tag_names, tag_colors))
            all_cls = set()
            for tag, _, _ in tags_nm:
                all_cls.update(all_found[tag].keys())

            if not all_cls:
                print('Жодна модель нічого не знайшла.')
            else:
                cls_list  = sorted(all_cls)
                cls_names = [CLASS_UKR_FULL.get(idx_to_name.get(ci,''),'?')
                             for ci in cls_list]
                n_cls = len(cls_list)
                x, width = np.arange(n_cls), 0.25

                fig, (ax_cnt, ax_conf) = plt.subplots(
                    1, 2, figsize=(max(10, n_cls * 3 + 2), 5)
                )
                fig.suptitle('Порівняння моделей - Multi-Scale Sliding Window',
                             fontsize=13, fontweight='bold')

                for i, (tag, nm, clr) in enumerate(tags_nm):
                    counts = [all_found[tag].get(ci, 0) for ci in cls_list]
                    bars   = ax_cnt.bar(x + i * width, counts, width,
                                        label=nm, color=clr,
                                        edgecolor='white', alpha=0.9)
                    for bar, v in zip(bars, counts):
                        if v > 0:
                            ax_cnt.text(
                                bar.get_x() + bar.get_width() / 2,
                                bar.get_height() + 0.1, str(v),
                                ha='center', va='bottom', fontsize=9
                            )

                ax_cnt.set_xticks(x + width)
                ax_cnt.set_xticklabels(cls_names, rotation=15, ha='right', fontsize=9)
                ax_cnt.set_ylabel('Кількість знахідок')
                ax_cnt.set_title('Знайдено зон по класах')
                ax_cnt.legend(fontsize=9); ax_cnt.grid(axis='y', alpha=0.3)
                ax_cnt.set_ylim(0, (ax_cnt.get_ylim()[1] or 1) * 1.25)

                for i, (tag, nm, clr) in enumerate(tags_nm):
                    confs = [all_avg_conf[tag].get(ci, 0) * 100 for ci in cls_list]
                    bars  = ax_conf.bar(x + i * width, confs, width,
                                        label=nm, color=clr,
                                        edgecolor='white', alpha=0.9)
                    for bar, v in zip(bars, confs):
                        if v > 0:
                            ax_conf.text(
                                bar.get_x() + bar.get_width() / 2,
                                bar.get_height() + 0.5, f'{v:.0f}%',
                                ha='center', va='bottom', fontsize=9
                            )

                ax_conf.set_xticks(x + width)
                ax_conf.set_xticklabels(cls_names, rotation=15, ha='right', fontsize=9)
                ax_conf.set_ylabel('Середня впевненість (%)')
                ax_conf.set_title('Впевненість по класах')
                ax_conf.axhline(y=60, color='red', linestyle='--', alpha=0.4, linewidth=1)
                ax_conf.legend(fontsize=9); ax_conf.grid(axis='y', alpha=0.3)
                ax_conf.set_ylim(0, 110)
                plt.tight_layout()
                plt.savefig('04_single_image_analysis.png', dpi=120, bbox_inches='tight')
                plt.show()

            print(f"\n  {'Модель':28s}  {'Клас':24s}  {'Зон':>5}  {'Впевн.':>8}")
            print('  ' + '-' * 70)
            for tag, nm, _ in tags_nm:
                found, avg_conf = all_found[tag], all_avg_conf[tag]
                if found:
                    first = True
                    for ci, cnt in found.items():
                        cls    = CLASS_UKR_FULL.get(idx_to_name.get(ci, ''), '?')
                        conf   = avg_conf.get(ci, 0) * 100
                        nm_str = nm if first else ''
                        print(f"  {nm_str:28s}  {cls:24s}  {cnt:>5}  {conf:>7.1f}%")
                        first  = False
                else:
                    print(f"  {nm:28s}  {'нічого не знайдено':24s}")
            print('  ' + '-' * 70)

        with report_out:
            clear_output(wait=True)
            lines = [
                '=' * 62,
                '  ЗВІТ АНАЛІЗУ - Multi-Scale Sliding Window',
                f'  Час: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
                f'  Шукали: {", ".join(CLASS_UKR_FULL.get(n,"?") for n in target_names)}',
                '=' * 62, '',
            ]
            for tag, nm in zip(tags, tag_names):
                found, avg_conf = all_found[tag], all_avg_conf[tag]
                lines.append(f'  {nm}:')
                if found:
                    for ci, cnt in found.items():
                        cls  = CLASS_UKR_FULL.get(idx_to_name.get(ci, ''), '?')
                        conf = avg_conf.get(ci, 0) * 100
                        lines.append(f'    {cls}: {cnt} зон, впевненість {conf:.1f}%')
                else:
                    lines.append('    нічого не знайдено')
                lines.append('')
            lines += ['=' * 62]
            report = '\n'.join(lines)
            with open('diploma_report.txt', 'a', encoding='utf-8') as f:
                f.write(report + '\n\n')
            print(report)

        run_status.value = (
            "<span style='color:#2ecc71;font-size:12px;'>Готово! Перегляньте вкладки.</span>"
        )
        tab.selected_index = 0

    except Exception as ex:
        run_status.value = f"<span style='color:#e74c3c;font-size:12px;'>Помилка: {ex}</span>"
        import traceback; traceback.print_exc()
    finally:
        run_btn.disabled    = False
        run_btn.description = 'Запустити аналіз'


run_btn.on_click(on_run)

ui = widgets.VBox([
    header_w,
    step1_lbl, mode_btn,
    upload_block, gen_block,
    sep_line(),
    step2_lbl, cb_row, cb_btns,
    sep_line(),
    widgets.HBox([step3_lbl, run_btn, run_status],
                 layout=widgets.Layout(align_items='center')),
    sep_line(),
    tab,
    sep_line(),
    dl_box,
], layout=widgets.Layout(padding='8px', max_width='1100px'))

display(ui)

In [ ]:
# Фінальний звіт
keys = ['siamese', 'resnet', 'mobilenet']

print(f"\n  {'Модель':32s}  {'mAP':>8}  {'Top-1':>8}  {'Час мс':>8}")
print('  ' + '-' * 60)
for k in keys:
    rv = bench[k]
    print(f"  {MODEL_NAMES[k]:32s}  {rv['mAP']:7.2f}%  {rv['top1']:7.2f}%  {rv['time']:7.2f}")

s  = bench['siamese']['mAP']
r_ = bench['resnet']['mAP']
m  = bench['mobilenet']['mAP']
print(f'\n  Перевага Siamese над ResNet-50:    +{s - r_:.2f}% mAP')
print(f'  Перевага Siamese над MobileNetV3:  +{s - m:.2f}% mAP')

with open('diploma_report.txt', 'a', encoding='utf-8') as f:
    f.write('\n' + '=' * 58 + '\n')
    f.write('  BENCHMARK SUMMARY\n')
    f.write('=' * 58 + '\n')
    for k in keys:
        rv = bench[k]
        f.write(
            f"  {MODEL_NAMES[k]:32s}  "
            f"mAP={rv['mAP']:.2f}%  "
            f"Top-1={rv['top1']:.2f}%  "
            f"{rv['time']:.2f}ms\n"
        )
    f.write(f'  Siamese vs ResNet:    +{s - r_:.2f}%\n')
    f.write(f'  Siamese vs MobileNet: +{s - m:.2f}%\n')
